In [6]:
#IMPORTS
import streamlit as st
from fileinput import filename
import pandas as pd
import hvplot.pandas
from path import Path
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from st_aggrid import GridOptionsBuilder, AgGrid, GridUpdateMode, DataReturnMode
import holoviews as hv
import numpy as np
import sys
import os
from pathlib import Path, WindowsPath, PureWindowsPath
sys.path.append(str(Path.cwd().parent))
from Intelligence import Utils, cloud_data_collector, DataFramer
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

In [7]:
## Welcome to Company Predictive Analytics
def GetSelectedTicker():
        ticker = 'NVDA'
        return ticker
    
ticker = GetSelectedTicker()
tickerdetails = None
TotalProfitLoss = 0


In [8]:
abspath = os.path.abspath("")
#print(abspath)
tickerdatapath1 = Path(abspath).parent
#print(tickerdatapath1)
location = tickerdatapath1 / 'TickerData' # type: ignore
#print(location)
dfTicker:pd.DataFrame = Utils.GetLocalTickerData(ticker, location)
#print(f'Loading ticker data for ticker {ticker}')
#print(dfTicker)
if(dfTicker is None):
    dfTicker = cloud_data_collector.GetYFinanceData(ticker, '2y')
    if(dfTicker is not None):
        dfTicker.drop('Adj Close', axis=1, inplace=True)
        Utils.SaveTickerData(ticker, dfTicker, location)

if(dfTicker is None):
    print(f"Error loading data for ticker {ticker}.")             
else:
    print(f'Loading ticker data for ticker {ticker}')
    print(dfTicker)


Loading ticker data for ticker NVDA
                  Open        High         Low       Close    Volume
Date                                                                
2021-10-04  205.039993  205.419998  195.550003  197.320007  34563500
2021-10-05  199.500000  206.479996  198.539993  204.509995  27928200
2021-10-06  201.199997  207.199997  200.800003  207.000000  29720200
2021-10-07  210.919998  213.220001  209.720001  210.750000  25691900
2021-10-08  211.009995  212.059998  207.750000  208.309998  15125800
...                ...         ...         ...         ...       ...
2023-09-27  423.299988  428.720001  416.290009  424.679993  44493500
2023-09-28  424.600006  434.459991  421.149994  430.890015  42466300
2023-09-29  438.269989  441.440002  433.070007  434.989990  39722100
2023-10-02  440.299988  451.750000  438.609985  447.820007  43329800
2023-10-03  448.079987  451.299988  432.459991  435.170013  46970700

[503 rows x 5 columns]


In [9]:
dfPlot = dfTicker.hvplot.line(
    height=600,
    rot=90,
    responsive=True
)
hv.render(dfPlot, backend='bokeh')
# Set the variables for short window and long window periods
sma9 = 9
sma20 = 20
sma50 = 50
sma100 = 100
sma200 = 200
print('Generate the SMAs')
dfTicker["SMA9"] = dfTicker["Close"].rolling(window=sma9).mean()
dfTicker["SMA20"] = dfTicker["Close"].rolling(window=sma20).mean()
dfTicker["SMA50"] = dfTicker["Close"].rolling(window=sma50).mean()
dfTicker["SMA100"] = dfTicker["Close"].rolling(window=sma100).mean()
dfTicker["SMA200"] = dfTicker["Close"].rolling(window=sma200).mean()
print(dfTicker)
print(f'dfTicker dtypes are {dfTicker.dtypes}')

Generate the SMAs
                  Open        High         Low       Close    Volume        SMA9       SMA20       SMA50      SMA100     SMA200
Date                                                                                                                           
2021-10-04  205.039993  205.419998  195.550003  197.320007  34563500         NaN         NaN         NaN         NaN        NaN
2021-10-05  199.500000  206.479996  198.539993  204.509995  27928200         NaN         NaN         NaN         NaN        NaN
2021-10-06  201.199997  207.199997  200.800003  207.000000  29720200         NaN         NaN         NaN         NaN        NaN
2021-10-07  210.919998  213.220001  209.720001  210.750000  25691900         NaN         NaN         NaN         NaN        NaN
2021-10-08  211.009995  212.059998  207.750000  208.309998  15125800         NaN         NaN         NaN         NaN        NaN
...                ...         ...         ...         ...       ...         ...      

In [10]:
lstDecimalCols = ['Open', 'High', 'Low', 'Close', 'SMA9', 'SMA20', 'SMA50', 'SMA100', 'SMA200']
dfTicker = DataFramer.SetDataFrameColsTo2Decimals(dfData=dfTicker, lstcols=lstDecimalCols)

print(dfTicker)
print(f'dfTicker dtypes are {dfTicker.dtypes}')

              Open    High     Low   Close    Volume    SMA9   SMA20   SMA50  SMA100  SMA200
Date                                                                                        
2021-10-04  205.04  205.42  195.55  197.32  34563500     nan     nan     nan     nan     nan
2021-10-05  199.50  206.48  198.54  204.51  27928200     nan     nan     nan     nan     nan
2021-10-06  201.20  207.20  200.80  207.00  29720200     nan     nan     nan     nan     nan
2021-10-07  210.92  213.22  209.72  210.75  25691900     nan     nan     nan     nan     nan
2021-10-08  211.01  212.06  207.75  208.31  15125800     nan     nan     nan     nan     nan
...            ...     ...     ...     ...       ...     ...     ...     ...     ...     ...
2023-09-27  423.30  428.72  416.29  424.68  44493500  425.39  449.26  450.55  416.61  318.84
2023-09-28  424.60  434.46  421.15  430.89  42466300  424.49  446.17  449.76  418.05  320.14
2023-09-29  438.27  441.44  433.07  434.99  39722100  423.97  443.24  